In [1]:
"""
Central configuration for the THCT-Net sign-language recognition pipeline.

All hyperparameters, paths, device setup, feature definitions, and
reproducibility seeding live here. Every other module imports from config.
"""
from __future__ import annotations

import random
from pathlib import Path

import numpy as np

SEED = 42
# ──────────────────────────────────────────────────────────────────────
# Label / split constants
# ──────────────────────────────────────────────────────────────────────
BACKGROUND_LABEL = "background"

# All users available in the dataset (used by leave-one-out).
ALL_USERS = ["user1", "user2", "user3", "user5"]

# Default users for single-split mode (override via CLI / main).
DEFAULT_DEV_USERS = ["user1", "user2", "user5"]
DEFAULT_TEST_USER = "user3"
DEV_VAL_RATIO = 0.12
DEV_VAL_SEED = SEED + 202

# ──────────────────────────────────────────────────────────────────────
# Training hyperparameters
# ──────────────────────────────────────────────────────────────────────
BATCH_SIZE = 4
GLOSS_BALANCED_GLOSSES_PER_BATCH = 4
GLOSS_BALANCED_SAMPLES_PER_GLOSS = 6
MODEL_NAME = "thct_net"
NORMALIZATION_NAME = "palm_ref"
EPOCHS = 7
LEARNING_RATE = 3e-4

# ──────────────────────────────────────────────────────────────────────
# Data Augmentation Configuration
# ──────────────────────────────────────────────────────────────────────
USE_AUGMENTATION = False
AUGMENT_ROTATION_PROB = 0.5
AUGMENT_ROTATION_RANGE = 8.0     # degrees (±8 deg)
AUGMENT_SCALING_PROB = 0.5
AUGMENT_SCALING_RANGE = (0.95, 1.05)
AUGMENT_NOISE_PROB = 0.5
AUGMENT_NOISE_STD = 2.0          # mm (since relative coordinates are in mm)
AUGMENT_DROPOUT_PROB = 0.0
AUGMENT_DROPOUT_RATE = 0.00


# ──────────────────────────────────────────────────────────────────────
# THCT-Net architecture
# ──────────────────────────────────────────────────────────────────────
D_MODEL = 128          # Transformer hidden dimension
NUM_HEADS = 4          # Attention heads
NUM_TRANSFORMER_LAYERS = 4   # Number of ISATA blocks
BASE_CH = 64           # CNN base channels
DROPOUT = 0.1          # Dropout rate (used in Transformer stream)

WINDOW_SIZE = 30
STRIDE = 1

# ──────────────────────────────────────────────────────────────────────
# Streaming / online decoder
# ──────────────────────────────────────────────────────────────────────
STREAM_MODE = "thct_net_batch"
WER_EXAMPLE_PRINT_COUNT = 5

LEAP_FPS = 30
MIN_SIGN_MS = 500
MIN_SIGN_FRAMES = max(1, int(round((MIN_SIGN_MS / 1000.0) * LEAP_FPS)))

BAG_SIZE = 5
BAG_AGGREGATION = "mean"
CONFIDENCE_THRESHOLD = 0.35
SIGN_BG_MARGIN = 0.10

ONLINE_WINDOW_SIZE = WINDOW_SIZE
ONLINE_STRIDE = 1

# ──────────────────────────────────────────────────────────────────────
# Feature definition (132 raw Leap features)
# ──────────────────────────────────────────────────────────────────────
HANDS = ["left", "right"]
FINGERS = ["thumb", "index", "middle", "ring", "pinky"]
BONES = ["metacarpal", "proximal", "intermediate", "distal"]
CARTESIAN_AXES = ["x", "y", "z"]
START_AXES = ["sx", "sy", "sz"]

FEATURE_KEYS: list[str] = []

# Palm and wrist coordinates per hand.
for _hand in HANDS:
    for _part in ["palm", "wrist"]:
        for _axis in CARTESIAN_AXES:
            FEATURE_KEYS.append(f"{_hand}_{_part}_{_axis}")

# Finger bone start-joint coordinates per hand.
for _hand in HANDS:
    for _finger in FINGERS:
        for _bone in BONES:
            for _axis in START_AXES:
                FEATURE_KEYS.append(f"{_hand}_{_finger}_{_bone}_{_axis}")

assert len(FEATURE_KEYS) == 132, f"Expected 132 features, got {len(FEATURE_KEYS)}"

INPUT_DIM = len(FEATURE_KEYS)

FEATURE_INDEX = {key: idx for idx, key in enumerate(FEATURE_KEYS)}

PALM_TRIPLETS: dict[str, tuple[int, int, int]] = {}
HAND_POSITION_TRIPLETS: dict[str, list[tuple[int, int, int]]] = {}

for _hand in HANDS:
    PALM_TRIPLETS[_hand] = tuple(
        FEATURE_INDEX[f"{_hand}_palm_{axis}"] for axis in CARTESIAN_AXES
    )

    _triplets: list[tuple[int, int, int]] = []
    _triplets.append(
        tuple(FEATURE_INDEX[f"{_hand}_wrist_{axis}"] for axis in CARTESIAN_AXES)
    )
    for _finger in FINGERS:
        for _bone in BONES:
            _triplets.append(
                tuple(FEATURE_INDEX[f"{_hand}_{_finger}_{_bone}_{_axis}"] for _axis in START_AXES)
            )
    HAND_POSITION_TRIPLETS[_hand] = _triplets



"""
Data loading: CSV/TXT parsing, recording discovery, and segment extraction.

Provides:
  - load_segments           : parse annotation TXT file
  - load_leap_csv           : parse Leap CSV → (T, 132) feature matrix
  - find_user_recordings    : discover matching CSV+TXT pairs per user
  - build_intervals_with_background : insert background gaps between signs
  - extract_segments_for_recording  : build per-interval records for one recording
  - load_all_segments       : full pipeline → segments_by_user dict
"""
from __future__ import annotations

import re
from collections import defaultdict
from pathlib import Path

import numpy as np
import pandas as pd
from tqdm.auto import tqdm


"""
Feature extraction and normalization for Leap Motion skeleton data.

Provides:
  - extract_features_from_row : CSV row → 132-D raw feature vector
  - palm_reference_normalize_frame : per-frame palm-reference normalization
  - palm_reference_normalize_sequence : vectorised version for (T, D) arrays
"""
from __future__ import annotations

import numpy as np
import pandas as pd



# ──────────────────────────────────────────────────────────────────────
# Helpers
# ──────────────────────────────────────────────────────────────────────

def _safe_float(value, default: float = 0.0) -> float:
    """Convert values to float safely; return default on invalid input."""
    try:
        if pd.isna(value):
            return default
        return float(value)
    except Exception:
        return default


# ──────────────────────────────────────────────────────────────────────
# Raw feature extraction
# ──────────────────────────────────────────────────────────────────────

def extract_features_from_row(row: pd.Series) -> np.ndarray:
    """Convert one Leap CSV row to one 132-D raw feature frame (no normalization)."""
    values = np.asarray(
        [_safe_float(row.get(key, 0.0)) for key in FEATURE_KEYS],
        dtype=np.float32,
    )
    return np.nan_to_num(values, nan=0.0, posinf=0.0, neginf=0.0).astype(
        np.float32, copy=False
    )


# ──────────────────────────────────────────────────────────────────────
# Palm-reference normalization
# ──────────────────────────────────────────────────────────────────────

def palm_reference_normalize_frame(frame: np.ndarray) -> np.ndarray:
    """Normalize one frame by subtracting each hand's palm from that hand's coordinates.

    Input shape: (132,)
    Output shape: (132,)
    """
    out = np.asarray(frame, dtype=np.float32).copy()
    if out.shape[0] != len(FEATURE_KEYS):
        raise ValueError(
            f"Expected frame with {len(FEATURE_KEYS)} features, got {out.shape[0]}"
        )

    for hand in HANDS:
        px, py, pz = PALM_TRIPLETS[hand]
        palm = out[[px, py, pz]].copy()
        if not np.all(np.isfinite(palm)):
            palm = np.zeros((3,), dtype=np.float32)

        for ix, iy, iz in HAND_POSITION_TRIPLETS[hand]:
            out[ix] = out[ix] - palm[0]
            out[iy] = out[iy] - palm[1]
            out[iz] = out[iz] - palm[2]

        # Keep palm channels for stable shape and explicit origin anchoring.
        out[px] = 0.0
        out[py] = 0.0
        out[pz] = 0.0

    return np.nan_to_num(out, nan=0.0, posinf=0.0, neginf=0.0).astype(
        np.float32, copy=False
    )


def palm_reference_normalize_sequence(sequence: np.ndarray) -> np.ndarray:
    """Apply palm-reference normalization frame-wise.

    Input shape: (T, D)
    Output shape: (T, D)
    """
    seq = np.asarray(sequence, dtype=np.float32)
    if seq.ndim != 2:
        raise ValueError(f"Expected sequence with shape (T, D), got {seq.shape}")
    if seq.shape[0] == 0:
        return seq.astype(np.float32, copy=False)

    normalized = np.empty_like(seq, dtype=np.float32)
    for i in range(seq.shape[0]):
        normalized[i] = palm_reference_normalize_frame(seq[i])
    return normalized



RECORDING_ID_PATTERN = re.compile(r"(P\d+_S\d+_R\d+)", re.IGNORECASE)


# ──────────────────────────────────────────────────────────────────────
# File loaders
# ──────────────────────────────────────────────────────────────────────

def load_segments(path: Path) -> list[dict]:
    """Load segmentation file into a list of {start, end, label} dicts."""
    path = Path(path)
    if not path.exists():
        return []

    df = pd.read_csv(
        path, sep=r"\s+", header=None,
        names=["start", "end", "label"], engine="python",
    )
    segments = []

    for _, row in df.iterrows():
        try:
            start = int(row["start"])
            end = int(row["end"])
            label = str(row["label"]).strip()
            if label:
                segments.append({"start": start, "end": end, "label": label})
        except Exception:
            continue

    return segments


def load_leap_csv(
    path: Path,
    return_dataframe: bool = False,
):
    """Load Leap CSV and return feature matrix (num_frames, 132)."""
    path = Path(path)
    if not path.exists():
        empty = np.zeros((0, len(FEATURE_KEYS)), dtype=np.float32)
        return (empty, pd.DataFrame()) if return_dataframe else empty

    df = pd.read_csv(path)
    if df.empty:
        empty = np.zeros((0, len(FEATURE_KEYS)), dtype=np.float32)
        return (empty, df) if return_dataframe else empty

    feature_rows = []
    for _, row in tqdm(
        df.iterrows(), total=len(df),
        desc=f"Extracting {path.name}", leave=False,
    ):
        feature_rows.append(extract_features_from_row(row))

    features = np.vstack(feature_rows).astype(np.float32)
    return (features, df) if return_dataframe else features


# ──────────────────────────────────────────────────────────────────────
# Recording discovery
# ──────────────────────────────────────────────────────────────────────

def extract_recording_id(filename: str) -> str | None:
    """Extract canonical recording ID from a CSV or TXT filename."""
    match = RECORDING_ID_PATTERN.search(filename)
    return match.group(1) if match else None


def find_user_recordings(dataset_root: Path) -> dict[str, list[dict]]:
    """Find matching (CSV, segmentation) pairs for each user directory."""
    dataset_root = Path(dataset_root)
    user_map: dict[str, list[dict]] = defaultdict(list)

    for user_dir in sorted(dataset_root.glob("user*")):
        if not user_dir.is_dir():
            continue

        leap_dir = user_dir / "leap_data"
        seg_dir = user_dir / "segmentation"
        if not leap_dir.exists() or not seg_dir.exists():
            continue

        seg_map = {}
        for seg_path in seg_dir.glob("*.txt"):
            recording_id = extract_recording_id(seg_path.name)
            if recording_id is not None:
                seg_map[recording_id] = seg_path

        for csv_path in leap_dir.glob("*.csv"):
            recording_id = extract_recording_id(csv_path.name)
            if recording_id is None:
                continue

            seg_path = seg_map.get(recording_id)
            if seg_path is not None:
                user_map[user_dir.name].append({
                    "recording_id": recording_id,
                    "csv_path": csv_path,
                    "seg_path": seg_path,
                })

    return user_map


# ──────────────────────────────────────────────────────────────────────
# Segment extraction
# ──────────────────────────────────────────────────────────────────────

def build_intervals_with_background(
    segment_defs: list[dict],
    num_frames: int,
    background_label: str = BACKGROUND_LABEL,
) -> list[dict]:
    """Build labeled intervals including background gaps between annotated signs."""
    if num_frames <= 0:
        return []

    cleaned = []
    for seg in segment_defs:
        try:
            start = max(0, int(seg["start"]))
            end = min(num_frames - 1, int(seg["end"]))
            label = str(seg["label"]).strip()
            if end >= start and label:
                cleaned.append({
                    "start": start, "end": end,
                    "label": label, "is_background": False,
                })
        except Exception:
            continue

    cleaned.sort(key=lambda x: (x["start"], x["end"]))
    intervals = []
    prev_end = -1

    for seg in cleaned:
        if seg["start"] > prev_end + 1:
            intervals.append({
                "start": prev_end + 1,
                "end": seg["start"] - 1,
                "label": background_label,
                "is_background": True,
            })

        intervals.append(seg)
        prev_end = max(prev_end, seg["end"])

    if prev_end < num_frames - 1:
        intervals.append({
            "start": prev_end + 1,
            "end": num_frames - 1,
            "label": background_label,
            "is_background": True,
        })

    return intervals


def extract_segments_for_recording(
    csv_path: Path,
    seg_path: Path,
) -> list[dict]:
    """Return per-interval records for one recording, including background gaps."""
    features, raw_df = load_leap_csv(csv_path, return_dataframe=True)
    segment_defs = load_segments(seg_path)

    if features.shape[0] == 0:
        return []

    interval_defs = build_intervals_with_background(
        segment_defs, num_frames=features.shape[0],
    )
    if not interval_defs:
        return []

    confidence_cols = [
        col for col in ["left_confidence", "right_confidence"]
        if col in raw_df.columns
    ]
    records = []

    for seg in interval_defs:
        start = max(0, int(seg["start"]))
        end = min(features.shape[0] - 1, int(seg["end"]))
        if end < start:
            continue

        segment_array = features[start : end + 1]
        if segment_array.size == 0:
            continue

        segment_confidence = None
        if confidence_cols:
            conf_slice = raw_df.loc[start:end, confidence_cols].to_numpy(
                dtype=np.float32,
            )
            if conf_slice.size > 0:
                segment_confidence = np.nanmean(conf_slice, axis=1)

        records.append({
            "segment": segment_array.astype(np.float32),
            "label": seg["label"],
            "confidence": segment_confidence,
            "segment_span": (start, end),
            "recording_features": features,
            "is_background": bool(seg.get("is_background", False)),
        })

    return records


# ──────────────────────────────────────────────────────────────────────
# High-level loader
# ──────────────────────────────────────────────────────────────────────

def load_all_segments(
    dataset_root: Path,
) -> tuple[dict[str, list[dict]], dict[str, list[dict]]]:
    """Discover all recordings and extract segments.

    Returns
    -------
    user_recordings : dict[user_name → list of recording metadata]
    segments_by_user : dict[user_name → list of segment dicts]
    """
    user_recordings = find_user_recordings(dataset_root)
    if not user_recordings:
        raise RuntimeError(
            "No matching Leap CSV and segmentation TXT pairs were found."
        )

    print("Matched recordings per user:")
    for user_name, recs in user_recordings.items():
        print(f"  {user_name}: {len(recs)} recordings")

    segments_by_user: dict[str, list[dict]] = defaultdict(list)
    for user_name, recordings in user_recordings.items():
        for rec in tqdm(recordings, desc=f"Extracting segments for {user_name}"):
            rec_segments = extract_segments_for_recording(
                rec["csv_path"], rec["seg_path"],
            )
            for item in rec_segments:
                item["recording_id"] = rec["recording_id"]
            segments_by_user[user_name].extend(rec_segments)

    total = sum(len(lst) for lst in segments_by_user.values())
    print(f"\nTotal raw segments (sign + background): {total}")

    return user_recordings, segments_by_user


c:\Shoab\ML\environments\pip_env\data_processing\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
import numpy as np
import pandas as pd

DATASET_ROOT = r"C:\Shoab\Thesis\Experiments\dataset"

user_recordings, segments_by_user = load_all_segments(DATASET_ROOT)


In [3]:

sign_records = []
bg_records = []

for user, segments in segments_by_user.items():
    for seg in segments:
        start, end = seg["segment_span"]
        length_frames = end - start + 1
        arr = seg["segment"]  # (T, 132)

        missing_frames = int(np.all(arr == 0.0, axis=1).sum()) if arr.size else 0
        missing_ratio = missing_frames / length_frames if length_frames else np.nan

        record = {
            "user": user,
            "recording_id": seg["recording_id"],
            "label": seg["label"],
            "start": start,
            "end": end,
            "length_frames": length_frames,
            "missing_frames": missing_frames,
            "missing_ratio": missing_ratio,
        }
        (bg_records if seg["is_background"] else sign_records).append(record)

sign_lengths_df = pd.DataFrame(sign_records)
bg_lengths_df = pd.DataFrame(bg_records)

sign_lengths_df["length_ms"] = sign_lengths_df["length_frames"] / LEAP_FPS * 1000
bg_lengths_df["length_ms"] = bg_lengths_df["length_frames"] / LEAP_FPS * 1000


# ── Per-class statistics ────────────────────────────────────────────
def summarize(df: pd.DataFrame, name: str) -> pd.DataFrame:
    stats = df.groupby("label").agg(
        count=("length_frames", "size"),
        min_frames=("length_frames", "min"),
        max_frames=("length_frames", "max"),
        mean_frames=("length_frames", "mean"),
        median_frames=("length_frames", "median"),
        p95_frames=("length_frames", lambda x: np.percentile(x, 95)),
        mean_missing_ratio=("missing_ratio", "mean"),
    ).sort_values("count", ascending=False)

    stats["mean_ms"] = stats["mean_frames"] / LEAP_FPS * 1000
    stats["median_ms"] = stats["median_frames"] / LEAP_FPS * 1000
    stats["p95_ms"] = stats["p95_frames"] / LEAP_FPS * 1000

    print(f"\n=== Per-class statistics: {name} ===")
    print(stats.to_string())
    return stats

sign_class_stats = summarize(sign_lengths_df, "signs (excl. background)")
bg_class_stats = summarize(bg_lengths_df, "background")


# ── Overall dataset summary ─────────────────────────────────────────
n_users = len(segments_by_user)
n_recordings = sum(len(v) for v in user_recordings.values())

class_counts = sign_lengths_df["label"].value_counts()
per_user_counts = sign_lengths_df.groupby("user")["label"].count()

total_sign_frames = sign_lengths_df["length_frames"].sum()
total_bg_frames = bg_lengths_df["length_frames"].sum()

summary = {
    "n_users": n_users,
    "n_recordings": n_recordings,
    "n_sign_classes": sign_lengths_df["label"].nunique(),
    "n_sign_instances": len(sign_lengths_df),
    "n_background_instances": len(bg_lengths_df),
    "total_frames_signs": int(total_sign_frames),
    "total_frames_background": int(total_bg_frames),
    "total_duration_signs_min": total_sign_frames / LEAP_FPS / 60,
    "total_duration_background_min": total_bg_frames / LEAP_FPS / 60,
    "shortest_sign_frames": int(sign_lengths_df["length_frames"].min()),
    "longest_sign_frames": int(sign_lengths_df["length_frames"].max()),
    "mean_sign_frames": sign_lengths_df["length_frames"].mean(),
    "median_sign_frames": sign_lengths_df["length_frames"].median(),
    "p95_sign_frames": np.percentile(sign_lengths_df["length_frames"], 95),
    "most_common_class": class_counts.idxmax(),
    "most_common_class_count": int(class_counts.max()),
    "least_common_class": class_counts.idxmin(),
    "least_common_class_count": int(class_counts.min()),
    "class_imbalance_ratio": class_counts.max() / class_counts.min(),
    "overall_missing_ratio_signs": sign_lengths_df["missing_frames"].sum() / total_sign_frames,
    "overall_missing_ratio_background": (
        bg_lengths_df["missing_frames"].sum() / total_bg_frames if total_bg_frames else np.nan
    ),
}

summary_df = pd.DataFrame([summary]).T.rename(columns={0: "value"})

print("\n=== Dataset summary ===")
print(summary_df.to_string())

print("\nInstances per user:")
print(per_user_counts.to_string())

print("\nClass distribution (sign instances per label):")
print(class_counts.to_string())

Matched recordings per user:
  user1: 76 recordings
  user2: 73 recordings
  user3: 55 recordings
  user5: 72 recordings


Extracting segments for user5: 100%|██████████| 72/72 [00:56<00:00,  1.27it/s]



Total raw segments (sign + background): 2293

=== Per-class statistics: signs (excl. background) ===
           count  min_frames  max_frames  mean_frames  median_frames  p95_frames  mean_missing_ratio      mean_ms    median_ms       p95_ms
label                                                                                                                                      
FARMING       58          40         192    76.741379           68.0      142.60            0.000000  2558.045977  2266.666667  4753.333333
BIG           58          34          90    60.120690           56.5       88.15            0.000000  2004.022989  1883.333333  2938.333333
SMALL         55          54         106    74.745455           74.0       93.00            0.000000  2491.515152  2466.666667  3100.000000
COME          53          35         112    65.301887           66.0       94.80            0.000000  2176.729560  2200.000000  3160.000000
VAN           53          34         125    64.226415     